In [13]:
import torch

torch.serialization.add_safe_globals([slice])

In [14]:
import logging
import os
import random
import time
import warnings
from collections import Counter
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

import ase.io
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from e3nn.nn import Gate
from e3nn.o3 import FullyConnectedTensorProduct, Irrep, Irreps, SphericalHarmonics
from mace.data import (
    AtomicData,
    Configuration,
    Configurations,
    KeySpecification,
    load_from_xyz,
)
from mace.modules import (
    EquivariantProductBasisBlock,
    LinearNodeEmbeddingBlock,
    RadialEmbeddingBlock,
)
from mace.modules.blocks import RealAgnosticResidualInteractionBlock
from mace.tools import AtomicNumberTable
from mace.tools.torch_geometric.dataloader import DataLoader
from numpy.typing import NDArray
from scipy.spatial import cKDTree
from sklearn.model_selection import train_test_split
from torch import Tensor, nn
from torch.nn import CrossEntropyLoss
from torch.nn.utils import clip_grad_norm_
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch_ema import ExponentialMovingAverage
from torch_geometric.data import Data
from torch_geometric.utils import scatter
from tqdm import tqdm

logging.disable(logging.WARNING)
warnings.filterwarnings("ignore")

In [15]:
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [16]:
class MACEClassifier(nn.Module):
    def __init__(
        self,
        r_max: float,
        num_bessel: int,
        num_polynomial_cutoff: int,
        max_ell: int,
        num_interactions: int,
        num_elements: int,
        hidden_irreps: Irreps,
        correlation: int,
        avg_num_neighbors: float,
        atomic_numbers: list[int],
        radial_MLP: list[int],
        mlp_out_dim: int,
        num_classes_int: int,
    ) -> None:
        super().__init__()

        self.register_buffer(
            "atomic_numbers", torch.tensor(atomic_numbers, dtype=torch.int64)
        )

        # Initial node embeddings: only 0e, think of it as nn.Embedding
        self.num_features = hidden_irreps.count(Irrep(0, 1))
        node_attrs_irreps = Irreps([(num_elements, (0, 1))])
        node_feats_irreps_init = Irreps([(self.num_features, (0, 1))])
        
        self.node_embedding = LinearNodeEmbeddingBlock(
            irreps_in=node_attrs_irreps,
            irreps_out=node_feats_irreps_init,
        )

        # Radial embedding: bessel encoding with envelope
        self.radial_embedding = RadialEmbeddingBlock(
            r_max=r_max,
            num_bessel=num_bessel,
            num_polynomial_cutoff=num_polynomial_cutoff,
        )
        edge_feats_irreps = Irreps([(self.radial_embedding.out_dim, (0, 1))])

        # Note that interaction_irreps is the intermediate 
        sh_irreps = Irreps.spherical_harmonics(max_ell)
        interaction_irreps = (sh_irreps * self.num_features).sort()[0].simplify() # type: ignore

        self.spherical_harmonics = SphericalHarmonics(
            sh_irreps, normalize=True, normalization="component"
        )

        # We use RealAgnosticInteractionBlock as our interaction portion
        self.interactions = nn.ModuleList()
        self.products = nn.ModuleList()

        for i in range(num_interactions):
            in_irreps = hidden_irreps if i > 0 else node_feats_irreps_init
            out_irreps = hidden_irreps if i < num_interactions - 1 else Irreps([(self.num_features, (0, 1))])

            interaction = RealAgnosticResidualInteractionBlock(
                node_attrs_irreps=node_attrs_irreps,
                node_feats_irreps=in_irreps,
                edge_attrs_irreps=sh_irreps,
                edge_feats_irreps=edge_feats_irreps,
                target_irreps=interaction_irreps,
                hidden_irreps=out_irreps,
                avg_num_neighbors=avg_num_neighbors,
                radial_MLP=radial_MLP
            )
            prod = EquivariantProductBasisBlock(
                node_feats_irreps=interaction_irreps,
                target_irreps=out_irreps,
                correlation=correlation,
                num_elements=num_elements,
                use_sc=True
            )
            self.interactions.append(interaction)
            self.products.append(prod)
        
        # Single head for classification (for now)
        total_scalar_dim = self.num_features * num_interactions
        
        self.classifier = nn.Sequential(
            nn.Linear(total_scalar_dim, mlp_out_dim),
            nn.SiLU(),
            nn.Linear(mlp_out_dim, num_classes_int),
        )

    def forward(self, data: dict[str, Tensor]) -> Tensor:
        # Prepare
        sources = data["edge_index"][0]
        dests = data["edge_index"][1]

        vectors = data["positions"][dests] - data["positions"][sources] + data["shifts"]
        lengths = torch.linalg.norm(vectors, dim=-1, keepdim=True)
        node_init_attrs = data["node_attrs"]
        edge_index = data["edge_index"]

        # Embeddings
        node_feats = self.node_embedding(node_init_attrs)
        edge_attrs = self.spherical_harmonics(vectors)
        edge_feats, cutoff = self.radial_embedding(
            lengths, node_init_attrs, edge_index, self.atomic_numbers
        )

        # Interaction
        scalar_feats_list: list[Tensor] = []

        for interaction, product in zip(self.interactions, self.products):
            node_feats, sc = interaction(
                node_attrs=node_init_attrs,
                node_feats=node_feats,
                edge_attrs=edge_attrs,
                edge_feats=edge_feats,
                edge_index=edge_index,
                cutoff=cutoff,
            )
            node_feats = product(
                node_feats=node_feats,
                sc=sc,
                node_attrs=node_init_attrs
            )

            # Important: scalars should all be at the front
            scalar_feats_list.append(node_feats[:, : self.num_features])
        
        node_inv_embs = torch.cat(scalar_feats_list, dim=-1)
        return self.classifier(node_inv_embs)

In [17]:
Z_TABLE = AtomicNumberTable([1])
R_MAX = 1.2
LABEL_MAP = {"bcc": 0, "cd": 1, "fcc": 2, "hcp": 3, "hd": 4, "sc": 5}
LABEL_TO_CRYSTAL = {v: k for k, v in LABEL_MAP.items()}

EPOCHS = 40
LR = 0.01
WEIGHT_DECAY = 5e-7
MAX_GRAD_NORM = 10.0
EMA_DECAY = 0.99
LABEL_SMOOTHING = 0.1

In [18]:
def load_dataset(
    path: Path, 
    z_table: AtomicNumberTable, 
    cutoff: float, 
    label_map: dict[str, int]
) -> list[AtomicData]:
    data_list: list[AtomicData] = []
    
    for crystal_type, label_num in tqdm(label_map.items()):
        crystal_dir = path / crystal_type
        for f in os.listdir(crystal_dir):
            _, configs = load_from_xyz(
                str(crystal_dir / f),
                key_specification=KeySpecification.from_defaults(),
                no_data_ok=True
            )
            for config in configs:
                atomic_data = AtomicData.from_config(
                    config=config,
                    z_table=z_table,
                    cutoff=cutoff
                )
                n_atoms = len(config.atomic_numbers)
                atomic_data.node_labels = torch.full((n_atoms,), label_num, dtype=torch.long)
                
                data_list.append(atomic_data)
    
    return data_list

In [19]:
dataset = load_dataset(Path("synthetic_data"), Z_TABLE, R_MAX, LABEL_MAP)

100%|██████████| 6/6 [00:01<00:00,  3.35it/s]


In [ ]:
set_seed()
train_data, test_data = train_test_split(dataset, test_size=0.2, random_state=42)

train_loader = DataLoader(train_data, batch_size=4, shuffle=True)
test_loader = DataLoader(test_data, batch_size=4, shuffle=False)

In [21]:
total_edges = 0
total_nodes = 0

for d in train_data:
    total_edges += d.edge_index.shape[1]
    total_nodes += d.positions.shape[0]

avg_num_neighbors = total_edges / total_nodes
print(f"Average neighbors: {avg_num_neighbors:.1f}")

Average neighbors: 21.7


In [ ]:
def make_optimizer(model: MACEClassifier, lr: float, weight_decay: float) -> Adam:
    return Adam(
        [
            {"name": "embedding", "params": model.node_embedding.parameters(), "weight_decay": 0.0},
            {"name": "interactions", "params": model.interactions.parameters(), "weight_decay": weight_decay},
            {"name": "products", "params": model.products.parameters(), "weight_decay": weight_decay},
            {"name": "classifier", "params": model.classifier.parameters(), "weight_decay": 0.0},
        ],
        lr=lr,
        amsgrad=True,
        betas=(0.9, 0.999),
    )


def train_epoch(
    model: MACEClassifier,
    loader: DataLoader,
    criterion: CrossEntropyLoss,
    optimizer: Adam,
    ema: ExponentialMovingAverage,
    device: torch.device,
) -> tuple[float, float]:
    model.train()
    total_loss = 0
    total_correct = 0
    total_atoms = 0

    for batch in tqdm(loader, desc="Training", leave=False):
        batch = batch.to(device)
        optimizer.zero_grad(set_to_none=True)

        logits = model(batch)
        labels = batch.node_labels
        loss = criterion(logits, labels)
        loss.backward()

        clip_grad_norm_(model.parameters(), max_norm=MAX_GRAD_NORM)
        optimizer.step()
        ema.update()

        n = labels.shape[0]
        total_loss += loss.item() * n
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total_atoms += n

    return total_loss / total_atoms, total_correct / total_atoms


@torch.inference_mode()
def evaluate(
    model: MACEClassifier,
    loader,
    criterion: CrossEntropyLoss,
    device: torch.device,
) -> tuple[float, float]:
    model.eval()
    total_loss = 0
    total_correct = 0
    total_atoms = 0

    for batch in tqdm(loader, desc="Evaluating", leave=False):
        batch = batch.to(device)
        logits = model(batch)
        labels = batch.node_labels

        n = labels.shape[0]
        total_loss += criterion(logits, labels).item() * n
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total_atoms += n

    return total_loss / total_atoms, total_correct / total_atoms


def train(model, train_loader, test_loader, device):
    set_seed()
    model = model.to(device)

    criterion = CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    optimizer = make_optimizer(model, lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = ReduceLROnPlateau(optimizer, factor=0.8, patience=10)
    ema = ExponentialMovingAverage(model.parameters(), decay=EMA_DECAY)

    best_test_acc = 0.0

    for epoch in range(1, EPOCHS + 1):
        train_loss, train_acc = train_epoch(
            model, train_loader, criterion, optimizer, ema, device
        )
        with ema.average_parameters():
            test_loss, test_acc = evaluate(model, test_loader, criterion, device)

        scheduler.step(test_loss)

        current_lr = optimizer.param_groups[0]["lr"]
        print(
            f"Epoch {epoch:03d} | "
            f"Train Loss={train_loss:.4f} Acc={train_acc:.4f} | "
            f"Test Loss={test_loss:.4f} Acc={test_acc:.4f} | "
            f"LR={current_lr:.2e}"
        )

        if test_acc > best_test_acc:
            best_test_acc = test_acc
            with ema.average_parameters():
                torch.save(model.state_dict(), "best_model.pt")

    print(f"\nBest test accuracy: {best_test_acc:.4f}")

In [23]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MACEClassifier(
    r_max=R_MAX,
    num_bessel=8,
    num_polynomial_cutoff=5,
    max_ell=2,
    num_interactions=2,
    num_elements=1,
    hidden_irreps=Irreps("32x0e + 32x1o"),
    correlation=2,
    avg_num_neighbors=avg_num_neighbors,
    atomic_numbers=[1],
    radial_MLP=[64, 64, 64],
    mlp_out_dim=64,
    num_classes_int=len(LABEL_MAP),
)

train(model, train_loader, test_loader, device)

Epoch 001 | Train Loss=1.6687 Acc=0.3260 | Test Loss=1.3511 Acc=0.7552 | LR=1.00e-02


Epoch 002 | Train Loss=1.1576 Acc=0.6493 | Test Loss=1.0247 Acc=0.7388 | LR=1.00e-02


Epoch 003 | Train Loss=0.8616 Acc=0.8667 | Test Loss=0.8980 Acc=0.7854 | LR=1.00e-02


Epoch 004 | Train Loss=0.7344 Acc=0.8836 | Test Loss=0.7300 Acc=0.9065 | LR=1.00e-02


Epoch 005 | Train Loss=0.6867 Acc=0.9305 | Test Loss=0.6824 Acc=0.9101 | LR=1.00e-02


...

Epoch 036 | Train Loss=0.4279 Acc=0.9996 | Test Loss=0.4266 Acc=0.9997 | LR=1.00e-02


Epoch 037 | Train Loss=0.4280 Acc=0.9996 | Test Loss=0.4262 Acc=0.9997 | LR=1.00e-02


Epoch 038 | Train Loss=0.4272 Acc=0.9996 | Test Loss=0.4259 Acc=0.9997 | LR=1.00e-02


Epoch 039 | Train Loss=0.4273 Acc=0.9997 | Test Loss=0.4256 Acc=0.9998 | LR=1.00e-02


Epoch 040 | Train Loss=0.4274 Acc=0.9996 | Test Loss=0.4253 Acc=0.9999 | LR=1.00e-02

Best test accuracy: 0.9999
